# Floci masterclass — live walkthrough

**What we will prove, in two acts:**

1. **Demo 1** — dropping a file in S3 can trigger real compute (Lambda)
   with no server, cron job, or polling loop involved, *if* three
   separate permissions are all wired correctly.
2. **Demo 2** — an AWS identity has *zero* permissions by default; only an
   explicit policy grants anything, and Floci enforces that just like
   real AWS when you turn it on.

**How this notebook relates to the repo's `demos/` scripts:**
[`deploy.py`](../demos/01-s3-lambda-notification/deploy.py) /
[`trigger_and_wait.py`](../demos/01-s3-lambda-notification/trigger_and_wait.py) /
[`run_demo.py`](../demos/02-iam-policy-enforcement/run_demo.py) are the
**repeatable deployment path** — what `make demo1`/`make demo2`, pytest and
CI actually run. This notebook is their **expanded teaching trace**: the
exact same AWS calls, unpacked one per cell with the "why", for a live
audience. Same handler code, same resource names — just told as a story
instead of automated.

**Before you start:** run `make up` from a terminal (once per session).
It starts Floci and this notebook's kernel container.

In [ ]:
import os
from pathlib import Path

# Euporie and `jupyter execute` may start the kernel in either the repo
# root or this notebook's own directory — normalize to the repo root so
# every relative path below (demos/...) resolves the same way either way.
if not Path("demos").exists():
    os.chdir(Path.cwd().parent)

Path.cwd()

## 0. Connect to Floci

In [ ]:
import json
import time
import urllib.parse
import urllib.request

import boto3
from botocore.exceptions import ClientError

ENDPOINT = os.environ.get("FLOCI_ENDPOINT", "http://floci:4566")

health = urllib.request.urlopen(f"{ENDPOINT}/_floci/health")
json.loads(health.read())["version"]

---
## Demo 1 — S3 upload triggers a Lambda

```
in/scores.csv  ──(S3 event)──▶  predict Lambda  ──(writes)──▶  out/scores.csv
   score                                                        score,prediction
   0.9                                                          0.9,1
   0.2                                                          0.2,0
```

**The story:** a scoring system drops a batch of rows in S3. The platform
enriches it — no application server to run, no cron job polling for new
files. The trigger *is* the upload.

### What does the handler assume is already true?

Before wiring any infrastructure, read the code that infrastructure exists
to support. It expects: an S3 event telling it what was uploaded, read
access to that object, and write access to write the result somewhere.

In [ ]:
print(Path("demos/01-s3-lambda-notification/lambda_predict/handler.py").read_text())

Three things this handler needs from the outside, each granted by a
*different* piece of IAM/S3 configuration — the part people most often
mix up:

```
Lambda service ──assumes──▶ execution role ──reads/writes──▶ S3 prefixes
S3 service     ──invokes──▶ Lambda function   (resource policy)
```

We'll set up each one, then watch the connection cross-check itself.

In [ ]:
iam = boto3.client("iam", endpoint_url=ENDPOINT)
lam = boto3.client("lambda", endpoint_url=ENDPOINT)
s3 = boto3.client("s3", endpoint_url=ENDPOINT)

ROLE_NAME = "lambda-exec-role"
FUNC_NAME = "predict"
BUCKET = "demo"

### Who may *become* the Lambda?

The trust policy: only the Lambda service is allowed to assume this role.
Nothing else — not `junior`, not you — can act as `predict` with it.

In [ ]:
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

try:
    role = iam.create_role(
        RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy)
    )
    role_arn = role["Role"]["Arn"]
except iam.exceptions.EntityAlreadyExistsException:
    role_arn = iam.get_role(RoleName=ROLE_NAME)["Role"]["Arn"]

role_arn

### What may the *running* Lambda do?

The execution policy: read `demo/in/*`, write `demo/out/*`, log to
CloudWatch. That's the whole footprint the handler needs — deliberately
not `AdministratorAccess`, so a bug in this function can't reach anything
outside those two prefixes.

In [ ]:
execution_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject"],
            "Resource": [f"arn:aws:s3:::{BUCKET}/in/*"],
        },
        {
            "Effect": "Allow",
            "Action": ["s3:PutObject"],
            "Resource": [f"arn:aws:s3:::{BUCKET}/out/*"],
        },
        {
            "Effect": "Allow",
            "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
            "Resource": "arn:aws:logs:*:*:*",
        },
    ],
}

iam.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName="predict-execution-policy",
    PolicyDocument=json.dumps(execution_policy),
)
print("Policy attached.")

### Package and ship the code

Same handler we just read, zipped exactly as Lambda expects (module at
the zip root).

In [ ]:
import zipfile

handler_path = Path("demos/01-s3-lambda-notification/lambda_predict/handler.py")
zip_path = Path("demos/01-s3-lambda-notification/predict.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(handler_path, "handler.py")

zip_bytes = zip_path.read_bytes()
f"{len(zip_bytes)} bytes"

In [ ]:
try:
    fn = lam.create_function(
        FunctionName=FUNC_NAME,
        Runtime="python3.12",
        Role=role_arn,
        Handler="handler.handler",
        Code={"ZipFile": zip_bytes},
        Timeout=60,
        MemorySize=256,
    )
    func_arn = fn["FunctionArn"]
except lam.exceptions.ResourceConflictException:
    lam.update_function_code(FunctionName=FUNC_NAME, ZipFile=zip_bytes)
    func_arn = lam.get_function(FunctionName=FUNC_NAME)["Configuration"]["FunctionArn"]

func_arn

Same lifecycle as real AWS Lambda: `Pending` while Floci provisions it,
then `Active`.

In [ ]:
for _ in range(30):
    conf = lam.get_function_configuration(FunctionName=FUNC_NAME)
    print(conf["State"])
    if conf["State"] == "Active":
        break
    time.sleep(2)

### The bucket the demo uploads into

In [ ]:
try:
    s3.create_bucket(Bucket=BUCKET)
except s3.exceptions.BucketAlreadyOwnedByYou:
    pass
print(f"Bucket '{BUCKET}' ready.")

### Who can *invoke* it?

A **resource-based** policy on the Lambda itself, separate from the
execution role above. Without this, S3's notification is silently
rejected by Lambda — this is the piece people most often forget, on real
AWS too.

In [ ]:
try:
    lam.add_permission(
        FunctionName=FUNC_NAME,
        StatementId="s3invoke",
        Action="lambda:InvokeFunction",
        Principal="s3.amazonaws.com",
        SourceArn=f"arn:aws:s3:::{BUCKET}",
    )
    print("Permission granted.")
except lam.exceptions.ResourceConflictException:
    print("Permission already present.")

### What event activates it?

In [ ]:
s3.put_bucket_notification_configuration(
    Bucket=BUCKET,
    NotificationConfiguration={
        "LambdaFunctionConfigurations": [
            {
                "LambdaFunctionArn": func_arn,
                "Events": ["s3:ObjectCreated:*"],
                "Filter": {"Key": {"FilterRules": [{"Name": "prefix", "Value": "in/"}]}},
            }
        ]
    },
)
print("Notification configured: in/* -> predict()")

Before triggering anything, look at what we just configured — the
relationship S3 will act on:

In [ ]:
s3.get_bucket_notification_configuration(Bucket=BUCKET)["LambdaFunctionConfigurations"]

---
### Prediction: what happens if we write to `in/scores.csv`?

Everything above was setup — four separate pieces of configuration that
all have to agree. **This cell is the only thing an end user would ever
do.**

In [ ]:
content = b"score\n0.9\n0.2\n0.7\n"

# Clear any leftover output from a previous run of this notebook so the
# timing below reflects this upload, not a stale file.
s3.delete_object(Bucket=BUCKET, Key="out/scores.csv")

upload_start = time.time()
s3.put_object(Bucket=BUCKET, Key="in/scores.csv", Body=content)
print("Uploaded in/scores.csv — watch for the Lambda to pick it up below.")

### Observer only: waiting for asynchronous evidence

The production path is purely event-driven — nothing polls in the
architecture itself. The loop below only exists so *we*, watching a
notebook, can see the asynchronous result land. On a cold Floci this can
take ~30s (pulling the Lambda runtime image the first time); it's fast
afterwards. Re-run this cell if it times out.

In [ ]:
TIMEOUT_SECONDS = 180
deadline = upload_start + TIMEOUT_SECONDS
result = None

while time.time() < deadline:
    try:
        obj = s3.get_object(Bucket=BUCKET, Key="out/scores.csv")
        result = obj["Body"].read().decode("utf-8")
        break
    except s3.exceptions.NoSuchKey:
        time.sleep(2)

elapsed = time.time() - upload_start
print(f"out/scores.csv appeared after {elapsed:.2f}s\n")
print(result if result else "TIMEOUT — re-run this cell.")

### So what?

- The `in/` prefix filter is why this doesn't loop forever: the Lambda
  writes to `out/`, which doesn't match its own trigger.
- The execution role can't touch anything outside `demo/in` and
  `demo/out` — a bug here can't reach other buckets or resources.
- Producer (whoever uploads) and processor (the Lambda) never call each
  other directly; S3 decouples them. Either side can change without the
  other knowing.

---
## Demo 2 — IAM policy enforcement

```
admin (test/test)  ──creates──▶  junior + policy
junior              ──attempts──▶  s3:ListAllMyBuckets

claim: identity alone grants nothing.
       only an explicit policy grants one specific action.
```

**Goal:** show that Floci actually *enforces* that claim — not just
records policies without checking them — once enforcement is turned on.

### Switch the emulator to enforcement mode

**Live pause — do this now, in a separate terminal:**

```bash
make floci-iam-on
```

This restarts Floci with `FLOCI_SERVICES_IAM_ENFORCEMENT_ENABLED=true`. A
cell in *this* notebook can't do it — this container has no Docker
access, by design (see the Security section of the README). Run
`make floci-iam-off` afterwards if you want to go back to the permissive
default for demo 1.

Once Floci is back up, confirm it answers again before continuing
(it drops connections for a few seconds while it restarts):

In [ ]:
health = urllib.request.urlopen(f"{ENDPOINT}/_floci/health")
json.loads(health.read())["version"]

### Admin client

`test`/`test` bypasses enforcement entirely — that's a Floci demo
convention for bootstrapping, **not** an AWS authorization pattern. Real
AWS has no equivalent "ignore my own IAM policies" credential.

In [ ]:
admin_iam = boto3.client(
    "iam", endpoint_url=ENDPOINT, aws_access_key_id="test", aws_secret_access_key="test"
)
USER = "junior"

### Reset `junior` to a clean slate

So this section behaves the same whether it's the first time you run it
today or the fifth: no leftover policy, no pile-up of old access keys.

In [ ]:
try:
    admin_iam.create_user(UserName=USER)
except admin_iam.exceptions.EntityAlreadyExistsException:
    pass

for policy_name in admin_iam.list_user_policies(UserName=USER)["PolicyNames"]:
    admin_iam.delete_user_policy(UserName=USER, PolicyName=policy_name)

for key in admin_iam.list_access_keys(UserName=USER)["AccessKeyMetadata"]:
    admin_iam.delete_access_key(UserName=USER, AccessKeyId=key["AccessKeyId"])

print(f"'{USER}' reset: no policies, no access keys.")

In [ ]:
key = admin_iam.create_access_key(UserName=USER)
junior_access_key = key["AccessKey"]["AccessKeyId"]
junior_secret_key = key["AccessKey"]["SecretAccessKey"]

s3_as_junior = boto3.client(
    "s3",
    endpoint_url=ENDPOINT,
    aws_access_key_id=junior_access_key,
    aws_secret_access_key=junior_secret_key,
)
junior_access_key  # secret key stays in the variable, not printed on screen

### Ask the room: 403, or a bucket list?

`junior` exists and has valid credentials — but no policy at all yet.
**This is the slide cell.**

In [ ]:
try:
    s3_as_junior.list_buckets()
    print("UNEXPECTED SUCCESS — is FLOCI_SERVICES_IAM_ENFORCEMENT_ENABLED=true? "
          "Run `make floci-iam-on` in a terminal, then re-run this cell.")
except ClientError as e:
    status = e.response["ResponseMetadata"]["HTTPStatusCode"]
    code_ = e.response["Error"]["Code"]
    message = e.response["Error"]["Message"]
    print(f"Expected denial observed: {status} {code_}")
    print(message)

### Grant exactly one permission

Not "give junior access" — one named action, on one resource scope.

In [ ]:
allow_list_buckets = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": "s3:ListAllMyBuckets", "Resource": "*"}
    ],
}

attach_start = time.time()
admin_iam.put_user_policy(
    UserName=USER, PolicyName="AllowListBuckets", PolicyDocument=json.dumps(allow_list_buckets)
)
print(json.dumps(allow_list_buckets, indent=2))
print("\nThis permits only bucket enumeration — it does not grant reading or")
print("writing any object. junior still can't touch demo/in or demo/out.")

### junior tries again

Expect success this time — and near-instant, unlike real AWS where policy
propagation can take a few seconds.

In [ ]:
resp = s3_as_junior.list_buckets()
elapsed = time.time() - attach_start
bucket_names = [b["Name"] for b in resp["Buckets"]]
print(f"Success {elapsed:.2f}s after attaching the policy.")
print(f"Buckets: {bucket_names}")
# Non-empty if you ran Demo 1 first in this session (that's expected —
# the point here is the 403 -> success transition, not an empty list).

### So what?

- Enforcement is **off by default** in Floci — without
  `FLOCI_SERVICES_IAM_ENFORCEMENT_ENABLED=true`, every call above would
  have succeeded even at the "403 expected" step. Worth calling out
  explicitly: this is a Floci convenience default, and the opposite of
  how real AWS behaves.
- Policy propagation was effectively instant here; real AWS can take a
  few seconds. Don't build a demo that depends on that delay existing.

---
## Cleanup

Not run automatically from here — tear down from a terminal when you're
done presenting:

```bash
make clean
```